<img src="images/logo.jpg" style="display: block; margin-left: auto; margin-right: auto;" alt="לוגו הקורס">

# <p style="text-align: right; direction: rtl; float: right;">Threading - הרצה מקבילית</p>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">מה זה Threading?</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
Threading מאפשר לתוכנית לעשות כמה דברים <strong>במקביל</strong> במקום בזה אחר זה.<br>
במקום לחכות שכל פעולה תסתיים לפני שמתחילים את הבאה, אפשר להריץ מספר פעולות בו-זמנית.
</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>דוגמאות לשימוש:</strong>
</p>

<ul style="text-align: right; direction: rtl; float: right; clear: both;">
    <li>הורדת מספר קבצים במקביל</li>
    <li>עיבוד נתונים בזמן שמציגים ממשק למשתמש</li>
    <li>שליחת מיילים למספר נמענים במקביל</li>
    <li>בדיקת מספר אתרים בו-זמנית</li>
</ul>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">דוגמה פשוטה - הבדל בין רגיל ל-Threading</p>

In [ ]:
import threading
import time

def task(name, duration):
    """משימה פשוטה שלוקחת זמן"""
    print(f"משימה {name} התחילה")
    time.sleep(duration)
    print(f"משימה {name} הסתיימה")

# הרצה רגילה - זה אחר זה
print("=== ללא Threading ===")
start = time.time()
task("A", 2)
task("B", 2)
print(f"סה\"כ: {time.time() - start:.1f} שניות\n")

# עם Threading - במקביל!
print("=== עם Threading ===")
start = time.time()

# יצירת threads
t1 = threading.Thread(target=task, args=("A", 2))
t2 = threading.Thread(target=task, args=("B", 2))

# התחלת הרצה
t1.start()
t2.start()

# המתנה לסיום
t1.join()
t2.join()

print(f"סה\"כ: {time.time() - start:.1f} שניות")

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>מה קרה?</strong>
</p>

<ul style="text-align: right; direction: rtl; float: right; clear: both;">
    <li><strong>ללא Threading:</strong> 2 + 2 = 4 שניות (משימה A מסתיימת, ואז משימה B מתחילה)</li>
    <li><strong>עם Threading:</strong> שתי המשימות רצו במקביל = ~2 שניות!</li>
</ul>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">המושגים החשובים</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong><code>threading.Thread()</code></strong> - יצירת thread חדש.
</p>

<ul style="text-align: right; direction: rtl; float: right; clear: both;">
    <li><code>target</code> - הפונקציה שתרוץ ב-thread</li>
    <li><code>args</code> - הפרמטרים לפונקציה (tuple)</li>
</ul>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong><code>.start()</code></strong> - מתחיל את הרצת ה-thread.
</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong><code>.join()</code></strong> - ממתין עד שה-thread מסתיים.
</p>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">דוגמה מעשית - הורדת קבצים במקביל</p>

In [ ]:
import os
import requests
import threading
import time

# יוצרים תיקייה להורדות (אם היא כבר קיימת - לא קורה כלום)
os.makedirs('downloads', exist_ok=True)


def download_image(url, filename):
    """הורדת תמונה מהאינטרנט. מחזירה True אם הצליח"""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        with open(filename, 'wb') as f:
            f.write(response.content)
        print(f"✅ {filename} הורד בהצלחה")
        return True
    except Exception as e:
        print(f"❌ שגיאה בהורדת {filename}: {e}")
        return False


# רשימת תמונות להורדה (picsum.photos נותן תמונה אקראית בכל כתובת)
urls = [
    (f"https://picsum.photos/200/300?random={מספר}", f"downloads/image{מספר}.jpg")
    for מספר in range(1, 7)
]

# --- הורדה רגילה: אחת אחרי השנייה ---
print("=== הורדה רגילה (בלי Threading) ===")
start = time.time()
for url, filename in urls:
    download_image(url, filename)
זמן_רגיל = time.time() - start
print(f"זמן הורדה רגילה: {זמן_רגיל:.1f} שניות\n")

# --- הורדה במקביל עם Threading ---
print("=== הורדה במקביל (עם Threading) ===")
start = time.time()
threads = []

# יוצרים thread לכל הורדה ומתחילים אותם
for url, filename in urls:
    t = threading.Thread(target=download_image, args=(url, filename))
    threads.append(t)
    t.start()

# ממתינים שכל ההורדות יסתיימו
for t in threads:
    t.join()
זמן_מקביל = time.time() - start
print(f"זמן הורדה מקבילית: {זמן_מקביל:.1f} שניות")

print(f"\n🎉 חסכנו כ-{זמן_רגיל - זמן_מקביל:.1f} שניות (פי {זמן_רגיל / זמן_מקביל:.1f} מהר יותר)")
print("הקבצים נמצאים בתיקייה downloads:", sorted(os.listdir('downloads')))

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>מה קרה כאן?</strong><br>
הרצנו את אותן 6 הורדות פעמיים: פעם אחת אחרי השנייה, ופעם אחת במקביל.<br>
כשהורדה אחת מחכה לתשובה מהשרת, ה-thread שלה "ישן" ו-threads אחרים יכולים להתקדם - לכן הזמן הכולל קצר בהרבה.<br>
<code>os.makedirs('downloads', exist_ok=True)</code> יוצרת את התיקייה, ואם היא כבר קיימת לא נקבל שגיאה.<br>
שימו לב: ההדפסות של ה-threads יכולות להתערבב בסדר שונה בכל הרצה, כי אנחנו לא יודעים איזו הורדה תסתיים ראשונה.<br>
המספרים משתנים לפי מהירות האינטרנט, אבל ההורדה המקבילית אמורה להיות מהירה יותר באופן ברור.
</p>

<div class="align-center" style="display: flex; text-align: right; direction: rtl; clear: both;">
    <div style="display: flex; width: 10%; float: right; clear: both;">
        <img src="images/tip.png" style="height: 50px !important;" alt="טיפ">
    </div>
    <div style="width: 90%">
        <p style="text-align: right; direction: rtl; float: right; clear: both;">
        <strong>שימו לב</strong>:<br>
            אם הקוד למעלה לא רץ, כנראה שאין לכם את הספרייה <code>requests</code>: <code>pip install requests</code>.<br>
אם אין חיבור לאינטרנט תראו הודעות ❌ - זה בסדר, התוכנית ממשיכה לעבוד ולא קורסת.
        </p>
    </div>
</div>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">Thread שרץ ברקע</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
אפשר להריץ ב-thread גם משימה שחוזרת על עצמה - למשל "בדוק משהו כל כמה שניות" - בזמן שהתוכנית הראשית ממשיכה לעבוד.<br>
זה הבסיס למערכות התראות. בדוגמה: משימת הרקע בודקת 3 פעמים, ובינתיים התוכנית הראשית עושה דברים אחרים.
</p>

In [ ]:
def בדיקה_תקופתית(מספר_בדיקות, מרווח_שניות, תוצאות):
    """מבצעת בדיקה כל כמה שניות ושומרת את התוצאות ברשימה משותפת"""
    for מספר in range(1, מספר_בדיקות + 1):
        print(f"   [רקע] בדיקה {מספר}")
        תוצאות.append(f"תוצאה {מספר}")
        time.sleep(מרווח_שניות)
    print("   [רקע] כל הבדיקות הסתיימו")


תוצאות_הרקע = []
t = threading.Thread(target=בדיקה_תקופתית, args=(3, 1, תוצאות_הרקע))
t.start()

# בזמן שהרקע עובד, התוכנית הראשית ממשיכה
for מספר in range(3):
    print(f"[ראשי] עובד על דברים אחרים... {מספר}")
    time.sleep(1)

t.join()  # מחכים שהרקע יסיים לפני שמשתמשים בתוצאות
print("התוצאות שנאספו:", תוצאות_הרקע)

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>מה קרה כאן?</strong><br>
ה-thread רץ במקביל לתוכנית הראשית: ראיתם הדפסות "[רקע]" ו-"[ראשי]" משולבות.<br>
העברנו ל-thread רשימה (<code>תוצאות_הרקע</code>) והוא הוסיף לה תוצאות - כך thread מחזיר מידע לתוכנית הראשית.<br>
<code>join()</code> חשוב: בלעדיו התוכנית הראשית עלולה להמשיך לפני שהרקע סיים.
</p>

<div class="align-center" style="display: flex; text-align: right; direction: rtl; clear: both;">
    <div style="display: flex; width: 10%; float: right; clear: both;">
        <img src="images/tip.png" style="height: 50px !important;" alt="טיפ">
    </div>
    <div style="width: 90%">
        <p style="text-align: right; direction: rtl; float: right; clear: both;">
        <strong>חשוב להבין</strong>:<br>
            threads עוזרים כשהתוכנית <strong>מחכה</strong> (רשת, קבצים, <code>sleep</code>).<br>
חישובים כבדים (מעבד) לא מתקצרים עם threads בפייתון בגלל ה-GIL - לשם כך משתמשים ב-<code>multiprocessing</code>, שלא נלמד בקורס.
        </p>
    </div>
</div>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">סיכום</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>מה למדנו?</strong>
</p>

<ul style="text-align: right; direction: rtl; float: right; clear: both;">
    <li><strong>Threading</strong> מאפשר הרצה מקבילית של מספר משימות</li>
    <li><strong>threading.Thread()</strong> יוצר thread חדש</li>
    <li><strong>.start()</strong> מתחיל הרצה</li>
    <li><strong>.join()</strong> ממתין לסיום</li>
    <li>השוואת זמנים: הורדה רגילה מול הורדה מקבילית</li>
    <li>thread שרץ ברקע ומחזיר תוצאות דרך רשימה</li>
</ul>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>מתי להשתמש?</strong>
</p>

<ul style="text-align: right; direction: rtl; float: right; clear: both;">
    <li>כשיש כמה משימות שאפשר לעשות במקביל</li>
    <li>הורדת קבצים, שליחת בקשות רשת</li>
    <li>כשיש המתנות ארוכות</li>
</ul>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
הבא בתור: <code>3_Asyncio_Basics</code> - נושא <strong>אופציונלי / העשרה</strong> שפותר את אותה בעיה בדרך אחרת.
</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>בהצלחה! 🚀</strong>
</p>